In [1]:
from experiments.bench import harness, plots
import pandas as pd
result = harness.load_result("exp2_pipeline")
print(result["environment"])

{'python': '3.12.13', 'platform': 'macOS-26.5.2-arm64-arm-64bit', 'machine': 'arm64', 'cpu_count': 10, 'cpu_model': 'Apple M4', 'ram': '16.0 GiB', 'cbc': 'Version: 2.10.3', 'neo4j_server': 'Neo4j Kernel 5.26.28 (community)', 'packages': {'numpy': '2.5.1', 'scipy': '1.18.0', 'pulp': '3.3.2', 'neo4j': '6.2.0', 'tiktoken': '0.13.0', 'openai': '2.52.0', 'pandas': '3.0.5'}}


In [2]:
summary = pd.DataFrame([{
    "model": result["data"]["model"],
    "full_llm_tokens": result["data"]["token_counts"]["full_llm"],
    "hybrid_tokens": result["data"]["token_counts"]["hybrid"],
    "full_llm_cost_usd": result["data"]["cost_usd"]["full_llm"],
    "hybrid_cost_usd": result["data"]["cost_usd"]["hybrid"],
    "savings_pct": result["data"]["savings_pct"],
}]).round(4)
display(summary)
display(pd.DataFrame(result["data"]["sensitivity"]).round(3))

,model,full_llm_tokens,hybrid_tokens,full_llm_cost_usd,hybrid_cost_usd,savings_pct
0,gpt-5-nano,87896,60292,0.2075,0.1423,31.4053


,freetext_multiplier,full_llm_tokens,hybrid_tokens,savings_pct
0,0.25,50305,22701,54.873
1,0.50,62926,35322,43.867
2,1.00,87896,60292,31.405
3,2.00,138007,110403,20.002
4,4.00,238229,210625,11.587


In [3]:
plots.exp2_savings(result)   # figures/exp2_savings.png 생성

PosixPath('/Users/honey/Dev/teamweaver/experiments/figures/exp2_savings.png')

In [4]:
from IPython.display import Markdown, display

# 최종 리뷰 Important 11(+ Critical 1 반영): 이 셀은 원래 markdown cell로 31.4%,
# 54.9%→11.6% 같은 수치를 손으로 타이핑해 두고 있었다. 이제 위 코드 셀에서 로드한
# result["data"]에서 직접 계산해 표시한다.
d = result["data"]
sens = sorted(d["sensitivity"], key=lambda s: s["freetext_multiplier"])
lo, hi = sens[0], sens[-1]
base = next((s for s in sens if abs(s["freetext_multiplier"] - 1.0) < 1e-9), None)
oms = {s["model"]: s for s in d.get("output_model_sensitivity", [])}
acc = d["accuracy"]
lat = d.get("latency", {})

lines = ["## 관찰 요약", ""]
base_mult_str = f"{base['freetext_multiplier']}×" if base else "기준"
lines.append(f"- 기준 페이로드(자유서술 배수 {base_mult_str})에서 절감률은 "
             f"**{d['savings_pct']:.1f}%**다 — 최초 가설 목표치는 `experiments/results/*.json` "
             "범위 밖의 로컬 설계 문서에만 있어 이 노트북에는 싣지 않는다.")
lines.append(f"- 민감도 곡선(`data.sensitivity`)은 자유서술 분량 배수 "
             f"{lo['freetext_multiplier']}×→{hi['freetext_multiplier']}×에 걸쳐 절감률이 "
             f"{lo['savings_pct']:.1f}%→{hi['savings_pct']:.1f}%로 단조 감소한다 — 절감률은 "
             "고정 상수가 아니라 정형/비정형 리뷰 내용 비중에 강하게 의존하는 값이다.")
if "output_proportional_to_input" in oms and "output_constant_across_arms" in oms:
    prop, const = oms["output_proportional_to_input"], oms["output_constant_across_arms"]
    lines.append("- savings_pct는 대수적으로 입력 토큰 비율에서만 나오며 출력 토큰 가정"
                 "(out_ratio)에는 불변이다"
                 "(`tests/test_exp2.py::test_savings_pct_is_independent_of_out_ratio`). 하지만 "
                 "출력 토큰 *모델* 자체를 바꾸면(출력이 입력에 비례 vs 두 arm에서 동일) 절감률은 "
                 f"{prop['savings_pct']:.1f}%에서 {const['savings_pct']:.1f}%까지 크게 달라진다 "
                 "(`data.output_model_sensitivity`) — 5.777이라는 비율 자체가 이 페이로드가 "
                 "아니라 hybrid-shaped parse 워크로드에서 측정된 값이므로, Full-LLM 페이로드로의 "
                 "적용은 다른 페이로드 형태 간 외삽이다.")
lines.append(f"- 정확도 축(`data.accuracy`)은 {acc['note']}. 라이브로 돌리더라도 구조적 문제가 "
             "남는다 — Full-LLM 페이로드는 정답 항목 선택을 입력 필드로 그대로 전달하므로, "
             "출력을 그 입력과 비교하면 동어반복적으로 높은 일치율이 나온다(Task 7 리포트 기록). "
             "정직한 재추출 오류율을 재려면 항목을 가린 별도 페이로드가 필요하다.")
if lat:
    status = "측정됨" if lat.get("measured") else "미측정"
    lines.append(f"- 지연(latency) 축(`data.latency`)은 **{status}** — {lat.get('note', '')}")

display(Markdown("\n".join(lines)))

## 관찰 요약

- 기준 페이로드(자유서술 배수 1.0×)에서 절감률은 **31.4%**다 — 최초 가설 목표치는 `experiments/results/*.json` 범위 밖의 로컬 설계 문서에만 있어 이 노트북에는 싣지 않는다.
- 민감도 곡선(`data.sensitivity`)은 자유서술 분량 배수 0.25×→4.0×에 걸쳐 절감률이 54.9%→11.6%로 단조 감소한다 — 절감률은 고정 상수가 아니라 정형/비정형 리뷰 내용 비중에 강하게 의존하는 값이다.
- savings_pct는 대수적으로 입력 토큰 비율에서만 나오며 출력 토큰 가정(out_ratio)에는 불변이다(`tests/test_exp2.py::test_savings_pct_is_independent_of_out_ratio`). 하지만 출력 토큰 *모델* 자체를 바꾸면(출력이 입력에 비례 vs 두 arm에서 동일) 절감률은 31.4%에서 0.8%까지 크게 달라진다 (`data.output_model_sensitivity`) — 5.777이라는 비율 자체가 이 페이로드가 아니라 hybrid-shaped parse 워크로드에서 측정된 값이므로, Full-LLM 페이로드로의 적용은 다른 페이로드 형태 간 외삽이다.
- 정확도 축(`data.accuracy`)은 live=False 이므로 미측정. 라이브로 돌리더라도 구조적 문제가 남는다 — Full-LLM 페이로드는 정답 항목 선택을 입력 필드로 그대로 전달하므로, 출력을 그 입력과 비교하면 동어반복적으로 높은 일치율이 나온다(Task 7 리포트 기록). 정직한 재추출 오류율을 재려면 항목을 가린 별도 페이로드가 필요하다.
- 지연(latency) 축(`data.latency`)은 **미측정** — 이번 실험은 라이브 타이밍 호출을 하지 않았다(비용 절감을 위해 오프라인 토큰 카운팅만 수행). 아래는 Task 6에서 실측된 파싱 워크로드 지연을 참고치로 인용한 것이며, 이번 실험의 Full-LLM/Hybrid 페이로드 형태로 직접 측정된 값이 아니다.